# 06 — Customer Segmentation & CLV Audit
**Goal**: Produce segments that are *meaningfully distinct and actionable* — not just statistically separable.
We run three complementary approaches:
1. **RFM segmentation** — Recency, Frequency, Monetary: the industry-standard baseline
2. **Behavioural K-Means** — clusters on behavioural + econometric features from notebook 04
3. **CLV gap analysis** — prove that existing CLV misranks members; build a `behavioral_clv` score
4. **OLS regression** — quantify how much behaviour vs demographics drives value (econometric layer)

**Input** : `data/processed/04_features.csv` + `data/processed/05_predictions.csv`
**Output** : `data/processed/06_segments.csv` — each member has an RFM label, K-Means cluster, behavioral_clv score, and churn probability

## 0. Imports & Config

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from pathlib import Path
import os, warnings
warnings.filterwarnings('ignore')

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score
import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor

PROC = Path('../data/processed')
FIG  = Path('../reports/figures')
os.makedirs(FIG, exist_ok=True)

KEY    = 'loyalty_number'
TARGET = 'churned'
CLV_COL = 'clv'

CHURN_COLOR   = '#C0392B'
RETAIN_COLOR  = '#2471A3'
SEGMENT_PALETTE = ['#2471A3','#1A8A4A','#E67E22','#8E44AD','#C0392B','#16A085']

sns.set_theme(style='whitegrid', font_scale=1.1)
plt.rcParams['figure.dpi'] = 120
print('Setup complete.')

---
## 1. Load Data

In [ ]:
features = pd.read_csv(PROC / '04_features.csv')
preds    = pd.read_csv(PROC / '05_predictions.csv')

df = features.merge(preds[[KEY, 'churn_prob', 'risk_bucket']], on=KEY, how='left')

# Resolve column names
flights_hist = next((c for c in ['total_flights_historical', 'total_flights_sum',
                                   'total_flights'] if c in df.columns), None)
pts_acc_col  = next((c for c in ['total_pts_accumulated','total_points_accumulated',
                                   'points_accumulated_sum'] if c in df.columns), None)
pts_red_col  = next((c for c in ['total_pts_redeemed','total_points_redeemed',
                                   'points_redeemed_sum'] if c in df.columns), None)
dist_col     = next((c for c in ['total_distance_historical','distance_sum',
                                   'distance'] if c in df.columns), None)

print(f'Dataset: {df.shape}')
print(f'Churn rate: {df[TARGET].mean():.1%}')
print(f'Columns resolved — flights: {flights_hist}, pts_acc: {pts_acc_col}')

---
## 2. RFM Segmentation — Industry Baseline
**Recency** = months since last flight (lower = better)
**Frequency** = total flights in history
**Monetary** = total distance or points accumulated (proxy for spend)

Each dimension scored 1–4 (quartile rank). RFM score = concatenation of three digits.

In [ ]:
# ── Build RFM scores ──────────────────────────────────────────────────────
rfm = df[[KEY, TARGET, 'churn_prob', CLV_COL]].copy()

# Recency: lower months_since = better score (reverse rank)
rfm['R_raw'] = df['months_since_last_flight'] if 'months_since_last_flight' in df.columns                else df.get('months_since_last_flight', 0)
rfm['R'] = pd.qcut(rfm['R_raw'], q=4, labels=[4,3,2,1], duplicates='drop').astype(int)

# Frequency
rfm['F_raw'] = df[flights_hist] if flights_hist else df.get('activity_rate', 0)
rfm['F'] = pd.qcut(rfm['F_raw'].rank(method='first'), q=4,
                    labels=[1,2,3,4], duplicates='drop').astype(int)

# Monetary: use CLV if available, else total points accumulated
if CLV_COL in df.columns:
    rfm['M_raw'] = df[CLV_COL]
elif pts_acc_col:
    rfm['M_raw'] = df[pts_acc_col]
else:
    rfm['M_raw'] = df.get('hyperbolic_flight_score', 0)

rfm['M'] = pd.qcut(rfm['M_raw'].rank(method='first'), q=4,
                    labels=[1,2,3,4], duplicates='drop').astype(int)

rfm['RFM_score'] = rfm['R'].astype(str) + rfm['F'].astype(str) + rfm['M'].astype(str)
rfm['RFM_total'] = rfm['R'] + rfm['F'] + rfm['M']

print(f'RFM score distribution (top 10):')
print(rfm['RFM_score'].value_counts().head(10))

In [ ]:
# ── Map RFM total score to actionable segment labels ─────────────────────
def rfm_segment(row):
    r, f, m = row['R'], row['F'], row['M']
    total = r + f + m
    if r == 4 and f >= 3 and m >= 3:  return 'Champions'
    if r >= 3 and f >= 3:             return 'Loyal'
    if r == 4 and f <= 2:             return 'Recent Low-Freq'
    if r == 1 and total >= 8:         return 'At-Risk High-Value'
    if r == 1 and total <= 5:         return 'Lost'
    if r == 2 and f >= 3:             return 'Needs Attention'
    if r >= 3 and f == 1:             return 'Promising New'
    return 'Hibernating'

rfm['rfm_segment'] = rfm.apply(rfm_segment, axis=1)

seg_summary = (
    rfm.groupby('rfm_segment')
    .agg(
        n_members   = (KEY,          'count'),
        churn_rate  = (TARGET,       'mean'),
        avg_clv     = (CLV_COL,      'mean'),
        avg_churn_p = ('churn_prob',  'mean')
    )
    .sort_values('churn_rate', ascending=False)
    .reset_index()
)
print('RFM segment summary:')
display(seg_summary.round(3))

In [ ]:
# ── RFM segment visualisation ─────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
segments  = seg_summary['rfm_segment'].tolist()
n_segs    = len(segments)
colors    = SEGMENT_PALETTE[:n_segs]

# Member count
axes[0].barh(seg_summary['rfm_segment'], seg_summary['n_members'],
             color=colors, alpha=0.85, edgecolor='white')
axes[0].set_xlabel('Members')
axes[0].set_title('RFM: member count per segment')
axes[0].invert_yaxis()
axes[0].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'{int(x):,}'))

# Churn rate
axes[1].barh(seg_summary['rfm_segment'], seg_summary['churn_rate'],
             color=colors, alpha=0.85, edgecolor='white')
axes[1].axvline(rfm[TARGET].mean(), color='gray', linestyle='--', lw=1.2,
                label='Overall avg')
axes[1].set_xlabel('Churn rate')
axes[1].set_title('RFM: churn rate per segment')
axes[1].invert_yaxis()
axes[1].xaxis.set_major_formatter(mticker.PercentFormatter(1.0))
axes[1].legend(fontsize=8)

# Average CLV
axes[2].barh(seg_summary['rfm_segment'], seg_summary['avg_clv'],
             color=colors, alpha=0.85, edgecolor='white')
axes[2].set_xlabel('Avg CLV ($)')
axes[2].set_title('RFM: average CLV per segment')
axes[2].invert_yaxis()
axes[2].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'${x:,.0f}'))

plt.suptitle('RFM segmentation overview', fontsize=13)
plt.tight_layout()
plt.savefig(FIG / '06_rfm_segments.png', bbox_inches='tight')
plt.show()

In [ ]:
# ── RFM scatter: Recency vs Frequency, sized by Monetary ─────────────────
fig, ax = plt.subplots(figsize=(10, 7))
for seg, color in zip(seg_summary['rfm_segment'], colors):
    sub = rfm[rfm['rfm_segment'] == seg]
    ax.scatter(
        sub['R_raw'], sub['F_raw'],
        s   = (sub['M_raw'] / sub['M_raw'].max() * 200 + 10).clip(10, 210),
        c   = color, alpha=0.45, label=f'{seg} (n={len(sub):,})',
        linewidths=0
    )
ax.set_xlabel('Recency (months since last flight — lower = more recent)')
ax.set_ylabel('Frequency (total flights)')
ax.set_title('RFM scatter — size = monetary value')
ax.legend(fontsize=8, bbox_to_anchor=(1.01, 1), loc='upper left')
plt.tight_layout()
plt.savefig(FIG / '06_rfm_scatter.png', bbox_inches='tight')
plt.show()

---
## 3. Behavioural K-Means Clustering
K-Means on the behavioural + econometric features surfaces patterns that RFM cannot see — specifically members who are *behaviourally disengaging* while still appearing valuable on historical metrics.

In [ ]:
# ── Select features for clustering ───────────────────────────────────────
# Use behavioural + econometric features — NOT demographic (avoid Simpson's paradox)
CLUSTER_FEATURES = [
    'hyperbolic_flight_score', 'recency_ratio', 'loss_aversion_score',
    'redemption_ratio', 'months_since_last_flight', 'flight_trajectory',
    'seasonal_volatility', 'flight_cv', 'longest_active_streak',
    'mean_fe_deviation', 'tenure_months', 'activity_rate'
]
CLUSTER_FEATURES = [f for f in CLUSTER_FEATURES if f in df.columns]
print(f'Clustering on {len(CLUSTER_FEATURES)} features:')
print(CLUSTER_FEATURES)

X_cluster = df[CLUSTER_FEATURES].fillna(0).copy()
X_cluster  = pd.DataFrame(
    StandardScaler().fit_transform(X_cluster),
    columns=CLUSTER_FEATURES
)

In [ ]:
# ── Elbow method + silhouette to choose K ────────────────────────────────
inertias    = []
silhouettes = []
K_range     = range(2, 9)

for k in K_range:
    km  = KMeans(n_clusters=k, random_state=42, n_init=10)
    lbl = km.fit_predict(X_cluster)
    inertias.append(km.inertia_)
    silhouettes.append(silhouette_score(X_cluster, lbl, sample_size=2000, random_state=42))
    print(f'  K={k}  inertia={km.inertia_:,.0f}  silhouette={silhouettes[-1]:.4f}')

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].plot(list(K_range), inertias, 'o-', color=CHURN_COLOR, linewidth=2)
axes[0].set_xlabel('Number of clusters (K)')
axes[0].set_ylabel('Inertia (within-cluster SSE)')
axes[0].set_title('Elbow method')

axes[1].plot(list(K_range), silhouettes, 's-', color=RETAIN_COLOR, linewidth=2)
axes[1].set_xlabel('Number of clusters (K)')
axes[1].set_ylabel('Silhouette score')
axes[1].set_title('Silhouette scores')
best_k = list(K_range)[int(np.argmax(silhouettes))]
axes[1].axvline(best_k, color='gray', linestyle='--', lw=1.2,
                label=f'Best K={best_k}')
axes[1].legend()

plt.suptitle('Choosing optimal K for behavioural clustering', fontsize=13)
plt.tight_layout()
plt.savefig(FIG / '06_elbow_silhouette.png', bbox_inches='tight')
plt.show()
print(f'\nBest K by silhouette: {best_k}')

In [ ]:
# ── Fit final K-Means with chosen K ──────────────────────────────────────
# You can override best_k here if the elbow suggests a different value
K_FINAL = best_k  # change this if needed, e.g. K_FINAL = 5

km_final = KMeans(n_clusters=K_FINAL, random_state=42, n_init=20)
df['cluster'] = km_final.fit_predict(X_cluster)

cluster_summary = (
    df.groupby('cluster')
    .agg(
        n_members        = (KEY,        'count'),
        churn_rate       = (TARGET,     'mean'),
        avg_churn_prob   = ('churn_prob','mean'),
        avg_clv          = (CLV_COL,    'mean'),
        avg_recency      = ('months_since_last_flight', 'mean'),
        avg_hyp_score    = ('hyperbolic_flight_score',  'mean'),
        avg_loss_aversion= ('loss_aversion_score',      'mean'),
        avg_trajectory   = ('flight_trajectory',        'mean')
    )
    .sort_values('churn_rate', ascending=False)
    .reset_index()
)

print(f'K-Means clusters (K={K_FINAL}):')
display(cluster_summary.round(3))

In [ ]:
# ── Assign human-readable cluster labels ─────────────────────────────────
# Read the cluster_summary above, then update this mapping to match your clusters
# The logic: high churn + high CLV = Silent Quitters; low churn + high freq = Champions

def label_cluster(row):
    cr  = row['churn_rate']
    clv = row['avg_clv']
    rec = row['avg_recency']
    hyp = row['avg_hyp_score']
    traj= row['avg_trajectory']

    global_churn = df[TARGET].mean()
    global_clv   = df[CLV_COL].mean() if CLV_COL in df.columns else 1

    if cr > global_churn * 1.5 and clv >= global_clv:
        return 'Silent Quitters'         # high value, high churn — TOP PRIORITY
    if cr > global_churn * 1.5 and clv < global_clv:
        return 'Low-Value Lapsers'       # low value, high churn — low priority
    if cr < global_churn * 0.5 and hyp > 0:
        return 'Loyal Champions'         # low churn, actively flying
    if cr < global_churn and rec > 3:
        return 'Stable Infrequent'       # low churn but not frequent flyers
    if traj < 0 and cr > global_churn:
        return 'Declining Actives'       # trajectory heading down — early warning
    return 'Moderate Risk'

cluster_summary['cluster_label'] = cluster_summary.apply(label_cluster, axis=1)
print('Cluster labels assigned:')
display(cluster_summary[['cluster','cluster_label','n_members','churn_rate','avg_clv']].round(3))

# Map labels back to main df
cluster_label_map = dict(zip(cluster_summary['cluster'], cluster_summary['cluster_label']))
df['cluster_label'] = df['cluster'].map(cluster_label_map)

In [ ]:
# ── PCA 2D visualisation of clusters ──────────────────────────────────────
pca       = PCA(n_components=2, random_state=42)
X_pca     = pca.fit_transform(X_cluster)
var_ratio = pca.explained_variance_ratio_

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# By cluster label
for i, (lbl, color) in enumerate(zip(cluster_summary['cluster_label'], SEGMENT_PALETTE)):
    mask = df['cluster_label'] == lbl
    axes[0].scatter(X_pca[mask, 0], X_pca[mask, 1],
                    c=color, label=f'{lbl} (n={mask.sum():,})',
                    alpha=0.35, s=10, linewidths=0)
axes[0].set_xlabel(f'PC1 ({var_ratio[0]:.1%} variance)')
axes[0].set_ylabel(f'PC2 ({var_ratio[1]:.1%} variance)')
axes[0].set_title('K-Means clusters — PCA 2D projection')
axes[0].legend(fontsize=7, markerscale=3)

# By churn status (overlay)
for churn_val, label, color in [(0,'Retained',RETAIN_COLOR),(1,'Churned',CHURN_COLOR)]:
    mask = df[TARGET] == churn_val
    axes[1].scatter(X_pca[mask, 0], X_pca[mask, 1],
                    c=color, label=label, alpha=0.2, s=8, linewidths=0)
axes[1].set_xlabel(f'PC1 ({var_ratio[0]:.1%} variance)')
axes[1].set_ylabel(f'PC2 ({var_ratio[1]:.1%} variance)')
axes[1].set_title('PCA space coloured by actual churn label')
axes[1].legend(markerscale=3)

plt.suptitle('Behavioural cluster structure (PCA 2D)', fontsize=13)
plt.tight_layout()
plt.savefig(FIG / '06_pca_clusters.png', bbox_inches='tight')
plt.show()

In [ ]:
# ── Cluster radar / heatmap profile ──────────────────────────────────────
# Normalise cluster means for comparison
profile_cols = [
    'hyperbolic_flight_score','recency_ratio','loss_aversion_score',
    'redemption_ratio','months_since_last_flight','flight_trajectory',
    'seasonal_volatility','tenure_months'
]
profile_cols = [c for c in profile_cols if c in df.columns]

cluster_profile = df.groupby('cluster_label')[profile_cols].mean()
# Normalise each column to 0-1 range for display
cluster_profile_norm = (cluster_profile - cluster_profile.min()) /                         (cluster_profile.max() - cluster_profile.min() + 1e-9)

fig, ax = plt.subplots(figsize=(14, 5))
sns.heatmap(
    cluster_profile_norm.T,
    annot=True, fmt='.2f', cmap='RdYlGn',
    linewidths=0.5, ax=ax, cbar_kws={'label': 'Normalised mean (0=low, 1=high)'}
)
ax.set_title('Cluster feature profiles — normalised means', fontsize=13)
ax.set_xlabel('Cluster label')
ax.set_ylabel('Feature')
plt.tight_layout()
plt.savefig(FIG / '06_cluster_heatmap.png', bbox_inches='tight')
plt.show()

---
## 4. CLV Gap Analysis — The Core Business Argument
**Hypothesis**: Existing CLV ranks members by *historical* value but ignores *forward-looking* behavioural signals.
We build a `behavioral_clv` score that discounts existing CLV by churn probability and weights it by recency.
The gap between CLV rank and behavioral_clv rank is the insight the CMO needs.

In [ ]:
# ── Build behavioral CLV ──────────────────────────────────────────────────
# behavioral_clv = CLV * (1 - churn_prob) * recency_weight
# Interpretation: expected future value accounting for disengagement risk

# Recency weight: members who flew recently retain more of their CLV
months_since = df['months_since_last_flight'] if 'months_since_last_flight' in df.columns                else pd.Series(6, index=df.index)

recency_decay = 1.0 / (1.0 + 0.15 * months_since)  # same hyperbolic function as nb04

df['behavioral_clv'] = (
    df[CLV_COL] *
    (1 - df['churn_prob']) *
    recency_decay
)

# ── CLV rank vs behavioral_clv rank ──────────────────────────────────────
df['clv_rank']          = df[CLV_COL].rank(ascending=False)
df['bclv_rank']         = df['behavioral_clv'].rank(ascending=False)
df['rank_gap']          = df['clv_rank'] - df['bclv_rank']
# Positive rank_gap = overvalued by CLV (CLV says important, behaviour says otherwise)
# Negative rank_gap = undervalued by CLV (behaviour says valuable, CLV doesn't see it)

n_members = len(df)
overvalued  = (df['rank_gap'] >  n_members * 0.10).sum()
undervalued = (df['rank_gap'] < -n_members * 0.10).sum()

print(f'CLV Gap Analysis:')
print(f'  Members overvalued  by CLV (rank_gap > 10%): {overvalued:,}')
print(f'  Members undervalued by CLV (rank_gap < -10%): {undervalued:,}')
print(f'  These {overvalued+undervalued:,} members would be mistargetted using CLV alone.')

In [ ]:
# ── Scatter: CLV rank vs behavioral_clv rank ─────────────────────────────
fig, ax = plt.subplots(figsize=(10, 7))

for churn_val, label, color in [(0,'Retained',RETAIN_COLOR),(1,'Churned',CHURN_COLOR)]:
    sub = df[df[TARGET]==churn_val]
    ax.scatter(sub['clv_rank'], sub['bclv_rank'],
               c=color, label=label, alpha=0.2, s=8, linewidths=0)

# Perfect agreement diagonal
ax.plot([1, n_members],[1, n_members], 'k--', lw=0.8, alpha=0.4, label='Perfect agreement')

# Annotate danger zone
ax.fill_between([1, n_members//4], [n_members*3//4, n_members],
                [n_members, n_members], alpha=0.06, color=CHURN_COLOR)
ax.text(n_members//8, n_members*0.93,
        'CLV says Top 25%\nBehaviour says Bottom 25%\n(Overvalued churners)',
        fontsize=8, color=CHURN_COLOR,
        bbox=dict(boxstyle='round,pad=0.3', facecolor='#FADBD8', alpha=0.7))

ax.set_xlabel('CLV rank (1 = highest CLV)')
ax.set_ylabel('Behavioral CLV rank (1 = highest behavioral CLV)')
ax.set_title('CLV rank vs Behavioral CLV rank\n— churners cluster in the overvalued zone')
ax.legend(markerscale=3)
plt.tight_layout()
plt.savefig(FIG / '06_clv_gap_scatter.png', bbox_inches='tight')
plt.show()

In [ ]:
# ── Who are the most overvalued members? ─────────────────────────────────
# Top 25% by CLV but bottom 50% by behavioral CLV — the "hidden at-risk" group
hidden_at_risk = df[
    (df['clv_rank']  <= n_members * 0.25) &
    (df['bclv_rank'] >= n_members * 0.50)
].copy()

print(f'Hidden at-risk members (top CLV but low behavioral CLV): {len(hidden_at_risk):,}')
print(f'  Actual churn rate in this group: {hidden_at_risk[TARGET].mean():.1%}')
print(f'  Overall churn rate             : {df[TARGET].mean():.1%}')
print(f'  -> These members are {hidden_at_risk[TARGET].mean()/df[TARGET].mean():.1f}x more likely to churn')
print(f'     than the overall base rate — yet CLV alone would NOT prioritise them.')

In [ ]:
# ── CLV gap distribution ─────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogram of rank gaps
axes[0].hist(df['rank_gap'], bins=60, color='steelblue', alpha=0.8, edgecolor='white')
axes[0].axvline(0, color='gray', linestyle='--', lw=1.2, label='No gap')
axes[0].axvline(df['rank_gap'].mean(), color=CHURN_COLOR, linestyle='-',
                lw=1.5, label=f'Mean gap: {df["rank_gap"].mean():.0f}')
axes[0].set_xlabel('Rank gap (CLV rank - Behavioral CLV rank)')
axes[0].set_ylabel('Members')
axes[0].set_title('Distribution of CLV rank gap')
axes[0].legend()

# Boxplot of rank gaps by risk bucket
bucket_order = ['Critical (>75%)','High (50-75%)','Medium (25-50%)','Low (<25%)']
bucket_data  = [df[df['risk_bucket']==b]['rank_gap'].dropna() for b in bucket_order
                if b in df['risk_bucket'].values]
present_buckets = [b for b in bucket_order if b in df['risk_bucket'].values]

axes[1].boxplot(bucket_data, labels=present_buckets, patch_artist=True,
                boxprops=dict(facecolor='lightblue', color='navy'),
                medianprops=dict(color=CHURN_COLOR, linewidth=2))
axes[1].axhline(0, color='gray', linestyle='--', lw=1)
axes[1].set_xlabel('Risk bucket')
axes[1].set_ylabel('CLV rank gap')
axes[1].set_title('CLV overvaluation by churn risk bucket')
axes[1].tick_params(axis='x', rotation=20)

plt.suptitle('CLV vs Behavioral CLV — gap analysis', fontsize=13)
plt.tight_layout()
plt.savefig(FIG / '06_clv_gap_distribution.png', bbox_inches='tight')
plt.show()

---
## 5. OLS Regression — What Actually Drives CLV?
**Econometric question**: Do demographic characteristics or behavioural signals explain more of the variance in CLV?
The regression coefficient magnitudes answer this — and give the CMO a data-backed argument for shifting strategy from demographic targeting to behavioural targeting.

In [ ]:
# ── Build OLS feature set ────────────────────────────────────────────────
OLS_FEATURES = {
    'behavioural': [
        'hyperbolic_flight_score', 'recency_ratio',
        'loss_aversion_score', 'redemption_ratio',
        'months_since_last_flight', 'tenure_months', 'activity_rate'
    ],
    'demographic': [
        'salary', 'tier_rank', 'marital_status', 'education', 'gender'
    ]
}
# Only keep columns that exist
all_ols_cols = []
for group, cols in OLS_FEATURES.items():
    OLS_FEATURES[group] = [c for c in cols if c in df.columns]
    all_ols_cols += OLS_FEATURES[group]

ols_df = df[[CLV_COL] + all_ols_cols].dropna().copy()

# Log-transform CLV (right-skewed)
ols_df['log_clv'] = np.log1p(ols_df[CLV_COL])

# ── Check VIF to detect multicollinearity ────────────────────────────────
X_ols = sm.add_constant(ols_df[all_ols_cols].astype(float))
vif_data = pd.DataFrame({
    'feature': X_ols.columns,
    'VIF'    : [variance_inflation_factor(X_ols.values, i)
                for i in range(X_ols.shape[1])]
}).sort_values('VIF', ascending=False)

print('VIF check (values > 10 indicate multicollinearity):')
display(vif_data[vif_data['VIF'] < 100].round(2))  # hide const VIF

high_vif = vif_data[(vif_data['VIF'] > 10) & (vif_data['feature'] != 'const')]
if len(high_vif) > 0:
    print(f'\nDropping high-VIF features: {high_vif["feature"].tolist()}')
    drop_vif = high_vif['feature'].tolist()
    all_ols_cols = [c for c in all_ols_cols if c not in drop_vif]

In [ ]:
# ── Fit OLS ───────────────────────────────────────────────────────────────
X_final = sm.add_constant(ols_df[all_ols_cols].astype(float))
y_ols   = ols_df['log_clv']

ols_result = sm.OLS(y_ols, X_final).fit()
print(ols_result.summary())

In [ ]:
# ── Decompose R² into behavioural vs demographic contribution ─────────────
behav_present = [c for c in OLS_FEATURES['behavioural'] if c in all_ols_cols]
demo_present  = [c for c in OLS_FEATURES['demographic']  if c in all_ols_cols]

def partial_r2(y, X_all_cols, subset_cols):
    """R² of full model minus R² of model without subset."""
    X_full    = sm.add_constant(ols_df[X_all_cols].astype(float))
    X_reduced = sm.add_constant(ols_df[[c for c in X_all_cols if c not in subset_cols]].astype(float))
    r2_full   = sm.OLS(y, X_full).fit().rsquared
    r2_red    = sm.OLS(y, X_reduced).fit().rsquared if X_reduced.shape[1] > 1 else 0
    return max(r2_full - r2_red, 0)

r2_behav = partial_r2(y_ols, all_ols_cols, behav_present)
r2_demo  = partial_r2(y_ols, all_ols_cols, demo_present)
r2_total = ols_result.rsquared

print(f'OLS R² decomposition:')
print(f'  Total R²                    : {r2_total:.4f}')
print(f'  Behavioural partial R²      : {r2_behav:.4f}  ({r2_behav/max(r2_total,0.001):.1%} of total)')
print(f'  Demographic partial R²      : {r2_demo:.4f}   ({r2_demo/max(r2_total,0.001):.1%} of total)')
print()
if r2_behav > r2_demo:
    print('  -> BEHAVIOURAL features explain MORE variance in CLV than demographics.')
    print('     This validates behaviour-led targeting over demographic segmentation.')
else:
    print('  -> Demographics explain more variance — still, behaviour adds meaningful signal.')

In [ ]:
# ── OLS coefficient plot ──────────────────────────────────────────────────
coef_df = pd.DataFrame({
    'feature'   : ols_result.params.index[1:],   # drop const
    'coef'      : ols_result.params.values[1:],
    'pvalue'    : ols_result.pvalues.values[1:],
    'ci_low'    : ols_result.conf_int().values[1:, 0],
    'ci_high'   : ols_result.conf_int().values[1:, 1]
}).sort_values('coef', ascending=True)

coef_df['significant'] = coef_df['pvalue'] < 0.05
coef_df['color'] = coef_df.apply(
    lambda r: CHURN_COLOR  if (r['coef'] < 0 and r['significant'])
    else RETAIN_COLOR if (r['coef'] > 0 and r['significant'])
    else '#BDC3C7', axis=1
)
coef_df['group'] = coef_df['feature'].apply(
    lambda f: 'Behavioural' if f in behav_present else 'Demographic'
)

fig, ax = plt.subplots(figsize=(10, 7))
bars = ax.barh(coef_df['feature'], coef_df['coef'],
               color=coef_df['color'], alpha=0.85, edgecolor='white')
ax.errorbar(coef_df['coef'], range(len(coef_df)),
            xerr=[coef_df['coef']-coef_df['ci_low'],
                  coef_df['ci_high']-coef_df['coef']],
            fmt='none', color='gray', capsize=3, linewidth=0.8)
ax.axvline(0, color='gray', linewidth=0.8, linestyle='--')
ax.set_xlabel('OLS coefficient (log CLV)')
ax.set_title('OLS regression: drivers of CLV (log-transformed)')

from matplotlib.patches import Patch
ax.legend(handles=[
    Patch(facecolor=RETAIN_COLOR, label='Positive, significant (p<0.05)'),
    Patch(facecolor=CHURN_COLOR,  label='Negative, significant (p<0.05)'),
    Patch(facecolor='#BDC3C7',    label='Not significant'),
], fontsize=8, loc='lower right')
plt.tight_layout()
plt.savefig(FIG / '06_ols_coefficients.png', bbox_inches='tight')
plt.show()

---
## 6. Segment × Churn Risk Matrix
The most actionable output: each cell tells the marketing team exactly *who* to target *and* how urgently.

In [ ]:
# ── Cross-tab: cluster label vs risk bucket ───────────────────────────────
bucket_order = ['Critical (>75%)','High (50-75%)','Medium (25-50%)','Low (<25%)']

matrix = (
    df.groupby(['cluster_label', 'risk_bucket'])
    .size()
    .reset_index(name='n')
    .pivot(index='cluster_label', columns='risk_bucket', values='n')
    .reindex(columns=[b for b in bucket_order if b in df['risk_bucket'].values])
    .fillna(0)
    .astype(int)
)

fig, ax = plt.subplots(figsize=(12, 6))
sns.heatmap(
    matrix,
    annot=True, fmt=',', cmap='YlOrRd',
    linewidths=0.5, ax=ax,
    cbar_kws={'label': 'Number of members'}
)
ax.set_title('Segment x Churn Risk Matrix — member counts', fontsize=13)
ax.set_xlabel('Churn risk bucket')
ax.set_ylabel('Behavioural segment')
plt.tight_layout()
plt.savefig(FIG / '06_segment_risk_matrix.png', bbox_inches='tight')
plt.show()

print('\nThis matrix drives the retention playbook in notebook 07.')
print('Focus on cells with HIGH member counts AND HIGH risk buckets.')

In [ ]:
# ── Business priority score ────────────────────────────────────────────────
# Priority = churn_prob * behavioral_clv (expected loss if member churns)
df['priority_score'] = df['churn_prob'] * df['behavioral_clv']
df['priority_rank']  = df['priority_score'].rank(ascending=False)

top_priority = (
    df[['priority_rank', KEY, 'cluster_label', 'risk_bucket',
        'churn_prob', CLV_COL, 'behavioral_clv', 'priority_score']]
    .sort_values('priority_rank')
    .head(20)
    .reset_index(drop=True)
)
top_priority.index += 1

print('Top 20 highest-priority members (expected value at risk):')
display(top_priority.round(2))

---
## 7. Full Segment Profile Summary

In [ ]:
final_profile = (
    df.groupby('cluster_label')
    .agg(
        n_members          = (KEY,              'count'),
        churn_rate         = (TARGET,           'mean'),
        avg_churn_prob     = ('churn_prob',      'mean'),
        avg_clv            = (CLV_COL,           'mean'),
        avg_behavioral_clv = ('behavioral_clv',  'mean'),
        avg_recency        = ('months_since_last_flight','mean'),
        avg_hyp_score      = ('hyperbolic_flight_score', 'mean'),
        avg_loss_aversion  = ('loss_aversion_score',     'mean'),
        avg_trajectory     = ('flight_trajectory',       'mean'),
        n_critical_risk    = ('risk_bucket',
                              lambda x: (x=='Critical (>75%)').sum())
    )
    .sort_values('avg_churn_prob', ascending=False)
    .reset_index()
)
final_profile['clv_gap']    = final_profile['avg_clv'] - final_profile['avg_behavioral_clv']
final_profile['pct_total']  = final_profile['n_members'] / len(df)

display(final_profile.round(2))

In [ ]:
# ── CLV vs behavioral_clv by segment bar chart ──────────────────────────
fig, ax = plt.subplots(figsize=(12, 5))

x     = np.arange(len(final_profile))
width = 0.35

bars1 = ax.bar(x - width/2, final_profile['avg_clv'],
               width, label='Avg CLV (existing)', color=RETAIN_COLOR, alpha=0.8)
bars2 = ax.bar(x + width/2, final_profile['avg_behavioral_clv'],
               width, label='Avg Behavioral CLV (new)', color=CHURN_COLOR, alpha=0.8)

ax.set_xticks(x)
ax.set_xticklabels(final_profile['cluster_label'], rotation=20, ha='right')
ax.set_ylabel('Average CLV ($)')
ax.set_title('CLV vs Behavioral CLV by segment\n— gap reveals misranked members')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v,_: f'${v:,.0f}'))
ax.legend()

# Annotate gap
for i, (clv, bclv) in enumerate(zip(final_profile['avg_clv'],
                                      final_profile['avg_behavioral_clv'])):
    gap = clv - bclv
    if abs(gap) > 100:
        ax.annotate(f'gap\n${gap:,.0f}',
                    xy=(i+width/2, max(clv, bclv)),
                    ha='center', fontsize=8, color='gray')

plt.tight_layout()
plt.savefig(FIG / '06_clv_vs_bclv_by_segment.png', bbox_inches='tight')
plt.show()

---
## 8. Save Outputs

In [ ]:
# ── Merge everything into final segments table ───────────────────────────
segments_out = df[[
    KEY, TARGET, 'churn_prob', 'risk_bucket', 'priority_score', 'priority_rank',
    CLV_COL, 'behavioral_clv', 'clv_rank', 'bclv_rank', 'rank_gap',
    'cluster', 'cluster_label',
    'rfm_segment' if 'rfm_segment' in df.columns else KEY
]].copy()

# Attach RFM if not already merged
if 'rfm_segment' not in segments_out.columns:
    segments_out = segments_out.merge(
        rfm[[KEY,'rfm_segment','RFM_score','RFM_total']], on=KEY, how='left'
    )
else:
    segments_out = segments_out.merge(
        rfm[[KEY,'RFM_score','RFM_total']], on=KEY, how='left'
    )

segments_out.to_csv(PROC / '06_segments.csv', index=False)
print(f'Saved: 06_segments.csv  ({segments_out.shape[0]:,} members)')

# ── Save segment profile for dashboard ───────────────────────────────────
final_profile.to_csv(PROC / '06_segment_profiles.csv', index=False)
print(f'Saved: 06_segment_profiles.csv')

# ── Save cluster model for inference in Streamlit ────────────────────────
import joblib
joblib.dump(km_final, '../models/kmeans_model.pkl')
print('Saved: models/kmeans_model.pkl')

In [ ]:
print('\n' + '='*60)
print('  SEGMENTATION COMPLETE — KEY NUMBERS FOR YOUR REPORT')
print('='*60)
print(f'  Members analysed        : {len(df):,}')
print(f'  RFM segments            : {rfm["rfm_segment"].nunique()}')
print(f'  K-Means clusters (K={K_FINAL})  : {df["cluster_label"].nunique()}')
print(f'  Hidden at-risk members  : {len(hidden_at_risk):,}')
print(f'  (Top CLV but low bCLV — would be missed by CLV-only targeting)')
print()
print(f'  OLS R² total            : {r2_total:.4f}')
print(f'  Behavioural partial R²  : {r2_behav:.4f}')
print(f'  Demographic partial R²  : {r2_demo:.4f}')
print()
print('  Top 3 at-risk segments by churn rate:')
for _, row in final_profile.head(3).iterrows():
    print(f'    {row["cluster_label"]:<25}: {row["churn_rate"]:.1%} churn, '
          f'avg CLV ${row["avg_clv"]:,.0f}, n={row["n_members"]:,}')
print('='*60)
print('\nNext step -> 07_retention_strategy.ipynb')